# Environment setup
The following code can help you install the packages required for the pyspark environment in Colab

In [ ]:
!pip install pyspark
!pip install nose

The following is to mount Google drive files

In [ ]:
# View and modify the working path
import os
from google.colab import drive

# View current working directory
print("Current Working Directory:", os.getcwd())

# Mount Google Drive
drive.mount('/content/gdrive')

# Change working directory to your file position
path = "/content/gdrive/My Drive/bdh-hw3-pyspark-publish_colab/code"
os.chdir(path)

# Confirm the change
print("Working Directory:", os.getcwd())


# Models

In [ ]:
import pandas as pd
class Medication:
    "medication class"
    def __init__(self, patientID, date, medicine):
        self.patientID = patientID
        self.date = date
        self.medicine = medicine

class LabResult:
    "lab class"
    def __init__(self, patientID, date, resultName, value):
        self.patientID = patientID
        self.date = date
        self.resultName = resultName
        self.value = value

class Diagnostic():
    "diagnostic class"
    def __init__(self, patientID, code, date):
        self.patientID = patientID
        self.date = date
        self.code = code

def create_initial_score_df():
    # Test names
    test_names = ['RDDTest', 'PhenotypeTest', 'FeatureConstructionTest','TestGetPurity','ClusteringTest']
    # Initial scores set to 0
    initial_scores = [0, 0, 0, 0, 0]

    # Creating the DataFrame
    df = pd.DataFrame({'Test': test_names, 'Score': initial_scores})
    return df

# Create the initial DataFrame
df_score = create_initial_score_df()

# Load RDD raw data [10 points]

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from datetime import datetime
from pyspark.sql.types import StringType, BooleanType, IntegerType, DoubleType
import sys
# sys.path.append('main/')
# from src.main.models import Diagnostic, Medication, LabResult

import logging
logging.getLogger('pyspark').setLevel(logging.ERROR)
logging.getLogger("py4j").setLevel(logging.ERROR)

# Change working directory base on your path
path = "/content/gdrive/My Drive/bdh-hw3-pyspark-publish_colab/code"
os.chdir(path)


# Define the case classes
# this is in the model part in original files

# these are from main file
def sql_date_parser(input, pattern="yyyy-MM-dd'T'HH:mm:ssX"):
    date_format = datetime.strptime(input, pattern)
    return date_format.date()

def load_rdd_raw_data(spark):

    '''
    TODO:  load the input .csv files in the datafolder as structured RDDs
    param:
            spark : the SparkSession, it is used to configure Spark, create DataFrames, and execute SQL queries.
    return:
            rdd format for the medication, lab result and diagnostic
    task:
            1. Define the SQL queries and convert the resulting DataFrames into RDDs.
            in the SQL:
                - lab result querying: The query is selecting four columns: Member_ID, Date_Resulted, Result_Name, and Numeric_Result from lab_results_INPUT
                                     with a condition that Numeric_Result is not null and not an empty string.
                - diagnostic querying: The query is selecting three columns: Member_ID, Encounter_DateTime, and Code_ID from encounter_dx_INPUT joined
                                     with encounter_INPUT on the common column Encounter_ID.
                - medication querying: The query is selecting three columns: Member_ID, Order_Date, and Drug_Name from medication_orders_INPUT.
            2. The structure of the RDDs corresponds to the Medication, LabResult, and Diagnostic classes, this should be the same format as the model/model.py.
               - The Medication RDD has fields: patientID, date, and medicine.
               - The LabResult RDD has fields: patientID, date, resultName, and value.
               - The Diagnostic RDD has fields: patientID, date and code.
    '''

    # you may modify the path base on yours
    relative_path = "/content/gdrive/My Drive/bdh-hw3-pyspark-publish_colab/code/data/"
    file_name = ["encounter_INPUT.csv",\
                "encounter_dx_INPUT.csv", \
                "lab_results_INPUT.csv", \
                 "medication_orders_INPUT.csv"]

    csv_files = [relative_path+x for x in file_name]

    for file in csv_files:
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file)
        table_name = file.split("/")[-1].split(".")[0]
        df.createOrReplaceTempView(table_name)
    medication_rdd = None
    lab_result_rdd = None
    diagnostic_rdd = None
    # start your code at here

    return medication_rdd, lab_result_rdd, diagnostic_rdd


if __name__ == '__main__':
    spark = SparkSession.builder.appName("myApp").getOrCreate()
    medication_rdd, lab_result_rdd, diagnostic_rdd = load_rdd_raw_data(spark)
    medication_rdd = spark.sparkContext.parallelize(medication_rdd.collect())
    print(medication_rdd.count())
    print(lab_result_rdd.count())
    print(diagnostic_rdd.count())
    print(type(medication_rdd))
    spark.stop()

In [ ]:
# run this block to get the Load RDD test result
import unittest
from pyspark.sql import SparkSession

class RDDTest(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder.appName("myApp").getOrCreate()
        cls.score = 0

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()
        if cls.score == 10:  # All tests passed
            print(f"All tests passed! Total Score: {cls.score}/10")
        else:
            print(f"Total Score: {cls.score}/10")

    def test_medication_rdd(self):
        try:
            numTrueMedication = 31552
            medication_rdd, _, _ = load_rdd_raw_data(self.spark)
            numMedication = medication_rdd.count()
            self.assertAlmostEqual(numTrueMedication, numMedication)
            RDDTest.score += 3
        except AssertionError:
            pass

    def test_lab_result_rdd(self):
        try:
            numTrueLabResult = 106894
            _, lab_result_rdd, _ = load_rdd_raw_data(self.spark)
            numLabResult = lab_result_rdd.count()
            self.assertAlmostEqual(numTrueLabResult, numLabResult)
            RDDTest.score += 3
        except AssertionError:
            pass

    def test_diagnostic_rdd(self):
        try:
            numTrueDiagnostic = 112811
            _, _, diagnostic_rdd = load_rdd_raw_data(self.spark)
            numDiagnostic = diagnostic_rdd.count()
            self.assertAlmostEqual(numTrueDiagnostic, numDiagnostic)
            RDDTest.score += 4
        except AssertionError:
            pass
if __name__ == '__main__':

    # Create a test suite
    suite = unittest.TestSuite()
    suite.addTests(unittest.TestLoader().loadTestsFromTestCase(RDDTest))

    # Run the test suite
    unittest.TextTestRunner().run(suite)
    # Store the score in a DataFrame
    df_score.loc[df_score['Test'] == 'RDDTest', 'Score'] = RDDTest.score

# Phenotype [25 points]

In [ ]:
from pyspark import SparkContext
from pyspark.rdd import RDD
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, lit, when, col, udf
from pyspark.sql.functions import min as min_

from typing import Tuple

import sys
sys.path.append('./')
# from src.main.models import Diagnostic, Medication, LabResult
# from src.main.loadRddRawData import load_rdd_raw_data


#class T2dmPhenotype:
T1DM_DX = {"250.01", "250.03", "250.11", "250.13", "250.21", "250.23", "250.31", "250.33", "250.41", "250.43",
               "250.51", "250.53", "250.61", "250.63", "250.71", "250.73", "250.81", "250.83", "250.91", "250.93"}

T2DM_DX = {"250.3", "250.32", "250.2", "250.22", "250.9", "250.92", "250.8", "250.82", "250.7", "250.72", "250.6",
               "250.62", "250.5", "250.52", "250.4", "250.42", "250.00", "250.02"}

T1DM_MED = {"lantus", "insulin glargine", "insulin aspart", "insulin detemir", "insulin lente", "insulin nph", "insulin reg", "insulin,ultralente"}

T2DM_MED = {"chlorpropamide", "diabinese", "diabanase", "diabinase", "glipizide", "glucotrol", "glucotrol xl",
                "glucatrol ", "glyburide", "micronase", "glynase", "diabetamide", "diabeta", "glimepiride", "amaryl",
                "repaglinide", "prandin", "nateglinide", "metformin", "rosiglitazone", "pioglitazone", "acarbose",
                "miglitol", "sitagliptin", "exenatide", "tolazamide", "acetohexamide", "troglitazone", "tolbutamide",
                "avandia", "actos", "actos", "glipizide"}

DM_RELATED_DX = {"790.21", "790.22", "790.2", "790.29", "648.81", "648.82", "648.83", "648.84", "648", "648",
    "648.01", "648.02", "648.03", "648.04", "791.5", "277.7", "V77.1", "256.4"}

abnormal_lab_values = {
        "HbA1c": 6.0,
        "Hemoglobin A1c": 6.0,
        "Fasting Glucose": 110.0,
        "Fasting blood glucose": 110.0,
        "fasting plasma glucose": 110.0,
        "Glucose": 110.0,
        "glucose": 110.0,
        "Glucose, Serum": 110.0
    }


def transform(medication, labResult, diagnostic, spark):
    """
    TODO: Transform given data set to a RDD of patients and corresponding phenotype
    param:
            medication: An RDD containing patient medication data.
            labResult: An RDD containing patient lab results.
            diagnostic: An RDD containing patient diagnostic data.
    return:
            phenotypeLabel: An RDD containing tuples. Each tuple contains a patient ID and a corresponding phenotype label.
                            the calss label value should be 1,2 and 3. 1 is the Case Patients, 2 is the  Control Patients, and 3 is the others

    hints:
            Use the provided hints as a roadmap to tackle each step of the transformation process systematically. Especially for the figure 1 and 2.
            Understand the criteria for labeling a patient with diabetes.
            There are some explains about the two roadmap
            Case Patients (from the first diagram):
                The flow starts with patients' diagnostic information.
                It filters out patients with specific diagnostic codes related to Type 1 Diabetes Mellitus (T1DM).
                The remaining patients are further filtered to identify those with diagnostic codes related to Type 2 Diabetes Mellitus (T2DM).
                The flow then uses medication information to filter out patients with T1DM medication orders.
                Among the remaining patients, those with T2DM medication orders are further categorized.
                There are additional steps related to medication order dates and other criteria to finalize the list of case patients.
            Control Patients (from the second diagram):
                The flow starts with lab result information related to glucose.
                It filters out patients based on specific abnormal lab values related to diabetes diagnosis.
                Among the remaining patients, those with specific diagnostic codes related to diabetes or related conditions are filtered out.
                The remaining patients are categorized as control patients.
            Other Patients:
                Patients who don't fall into the "case" or "control" categories based on the above criteria are categorized as other patients.

    Tips:
            1. Case sensitivity. Some data items contain uppercase and lowercase letters. Pay attention to how to deal with them.
            2. ‘try/except’ blocks allowed the tests to pass even if they were failing, 
                so maybe you can remove try/except blocks to provide more details about what was wrong when you debug it (For example, add a print statement to see where you are missing out). 
                But you MUST change the ‘try/except’ block back to their original version when you submit your code.
            3. Do not forget to include '250.*' (in DM_RELATED_DX.csv) in your filtering criteria.
            
    """
    phenotypeLabel = None
    # start your code at here


    return phenotypeLabel


if __name__ == '__main__':
    spark = SparkSession.builder.appName("myApp").getOrCreate()
    medication_rdd, lab_result_rdd, diagnostic_rdd = load_rdd_raw_data(spark)
    data = lab_result_rdd.collect()
    phenotypeLabel = transform(medication_rdd, lab_result_rdd, diagnostic_rdd, spark)
    print(phenotypeLabel.count())
    print(phenotypeLabel.filter(lambda x: x[1] == 1).map(lambda x: x[0]).count())
    print(phenotypeLabel.filter(lambda x: x[1] == 2).map(lambda x: x[0]).count())
    print(phenotypeLabel.filter(lambda x: x[1] == 3).map(lambda x: x[0]).count())
    spark.stop()


In [ ]:
# run this block to get the Phenotype test result
import unittest
from pyspark.sql import SparkSession

class PhenotypeTest(unittest.TestCase):
    score = 0

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder.appName('Feature Construction Test').getOrCreate()
        cls.patient_features = cls.spark.sparkContext.parallelize([
            (("patient1", "code2"), 49.0),
            (("patient1", "code7"), 19.0),
            (("patient1", "code1"), 24.0)
        ])

        # Load RDDs only once for all tests
        cls.medication_rdd, cls.lab_result_rdd, cls.diagnostic_rdd = load_rdd_raw_data(cls.spark)
        # cls.phenotypeLabel = transform(cls.medication_rdd, cls.lab_result_rdd, cls.diagnostic_rdd)
        cls.phenotypeLabel = transform(cls.medication_rdd, cls.lab_result_rdd, cls.diagnostic_rdd, cls.spark)

    def test_phenotype_cases(self):
        try:
            numTrueCases = 976
            numCases = self.phenotypeLabel.filter(lambda x: x[1] == 1).count()
            self.assertAlmostEqual(numTrueCases, numCases)
            PhenotypeTest.score += 8
        except AssertionError:
            pass

    def test_phenotype_control(self):
        try:
            numTrueControl = 948
            numControl = self.phenotypeLabel.filter(lambda x: x[1] == 2).count()
            self.assertAlmostEqual(numTrueControl, numControl)
            PhenotypeTest.score += 8
        except AssertionError:
            pass

    def test_phenotype_others(self):
        try:
            numTrueCases = 976
            numTrueControl = 948
            numTrueOthers = 3688 - numTrueCases - numTrueControl
            numOthers = self.phenotypeLabel.filter(lambda x: x[1] == 3).count()
            self.assertAlmostEqual(numTrueOthers, numOthers)
            PhenotypeTest.score += 9
        except AssertionError:
            pass

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()
        # print(f"Total Score: {PhenotypeTest.score}/25")
        if PhenotypeTest.score == 25:  # All tests passed
            print(f"All tests passed! Total Score: {PhenotypeTest.score}/25")
        else:
            print(f"Total Score: {PhenotypeTest.score}/25")
if __name__ == '__main__':
    # Create a test suite
    suite = unittest.TestLoader().loadTestsFromTestCase(PhenotypeTest)

    # Run the test suite
    unittest.TextTestRunner().run(suite)
    df_score.loc[df_score['Test'] == 'PhenotypeTest', 'Score'] = PhenotypeTest.score

# Feature construction  [17 points]

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors
from pyspark import RDD

from typing import Tuple
from typing import Set
from datetime import date

FeatureTuple = Tuple[Tuple[str, str], float]



def constructDiagnosticFeatureTuple(diagnostic: RDD[Diagnostic], candidateCode: Set = None) -> RDD[FeatureTuple]:
    '''
    TODO: Aggregate feature tuples from diagnostic with COUNT aggregation,

    param:
            diagnostic: an RDD containing diagnostic data
            candidateCode: a set of ICD9 codes to filter by
    return:
            diag: an RDD containing feature tuples
    hint:
        the tuple's first element is the patient ID, and the second element is the ICD9 code. For each occurrence, assign a value of 1 and accumulate it,
        which seems like ((patientID, code), count) 


    '''
    diag = None
    # start your code at here

    return diag

def constructMedicationFeatureTuple(medication: RDD[Medication], candidateMedication: Set = None) -> RDD[FeatureTuple]:
    '''
    TODO: Aggregate feature tuples from medication with COUNT aggregation,
    param:
            medication: an RDD containing medication data
            candidateMedication: a set of medication names to filter by
    return:
            med: a set of medication names to filter by
    hint:
            similar to the former one, which seems like ((patientID, medicine), count)
    '''
    med = None
    # start your code at here

    return med

def constructLabFeatureTuple(labResult: RDD[LabResult], candidateLab: Set = None) -> RDD[FeatureTuple] :
    '''
    TODO: Aggregate feature tuples from lab result, using AVERAGE aggregation
    param:
            labResult: an RDD containing lab result data.
            candidateLab: a set of lab result names to filter by
    return:
            lab: an RDD containing feature tuples
    hint:
            unlike the previous two functions,
            this one requires you to track the number of times a lab test was conducted, the value of each test, and compute the average value,
            which seems like ((patientID, testName), averageValue)
    '''
    lab = None
    # start your code at here

    return lab


def construct(feature):
    '''
    TODO: Given a feature tuples RDD, construct features in vector
          format for each patient. feature name should be mapped
          to some index and convert to sparse feature format.
    param:
            feature: an RDD containing tuples where the inner tuple's first element is a combination of patient ID and feature name,
                      and the second element represents the feature's value.
    return:
            result: an RDD where each entry is a patient ID paired with a sparse vector representation of their features
    hint:
            the function should process the input data to represent it in a sparse vector format, grouping by patient ID.
            the sparse vector format is beneficial when dealing with high-dimensional data where many values can be zero,
            as it saves memory by only storing non-zero values
    '''
    result = None
    # start your code at here

    return result



if __name__ == '__main__':
    spark = SparkSession.builder.appName('Construct Features').getOrCreate()

    sc = spark.sparkContext

    medication_rdd, lab_result_rdd, diagnostic_rdd = load_rdd_raw_data(spark)
    print(lab_result_rdd.count())
    diagnostic_feature_tuples = constructDiagnosticFeatureTuple(diagnostic_rdd)
    print(diagnostic_feature_tuples.count())
    medicine_feature_tuples = constructMedicationFeatureTuple(medication_rdd)
    print(medicine_feature_tuples.count())
    lab_result_feature_tuples  = constructLabFeatureTuple(lab_result_rdd)
    print(lab_result_feature_tuples.count())
    data = [Medication("patient1", date.today(), "code1"),
        Medication("patient1", date.today(), "code2")]
    meds = spark.sparkContext.parallelize(data)
    print(meds.collect())
    feature_rdd = constructMedicationFeatureTuple(meds)
    print(feature_rdd.collect())
    feature_sparse_vector = construct(feature_rdd)
    spark.stop()

In [ ]:
import unittest
from pyspark.sql import SparkSession
from datetime import date
from pyspark.ml.linalg import Vectors


class FeatureConstructionTest(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder.appName("Feature Construction Test").getOrCreate()
        cls.spark.sparkContext.setLogLevel("ERROR")
        cls.score = 0


    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()
        if FeatureConstructionTest.score == 17:  # All tests passed
            print(f"All tests passed! Total Score: {FeatureConstructionTest.score}/17")
        else:
            print(f"Total Score: {cls.score}/17")

    def test_unique_ids(self):
        try:
            patient_features = self.spark.sparkContext.parallelize([
                (("patient1", "code2"), 49.0),
                (("patient1", "code7"), 19.0),
                (("patient1", "code1"), 24.0)
            ])
            temp = construct(patient_features).collect()
            expected = ('patient1', Vectors.sparse(3, [(0, 24.0), (1, 49.0), (2, 19.0)]))
            self.assertEqual(temp[0], expected, "feature (event) ids are missed/repeated/unsorted")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass

    def test_sparse_vectors(self):
        try:
            patient_features = self.spark.sparkContext.parallelize([
                (("patient1", "code0"), 42.0),
                (("patient1", "code2"), 24.0),
                (("patient2", "code1"), 12.0)
            ])
            temp = construct(patient_features).sortBy(lambda x: x[0]).collectAsMap()
            expected = {
                "patient1": Vectors.sparse(3, [(0, 42.0), (2, 24.0)]),
                "patient2": Vectors.sparse(3, [(1, 12.0)])
            }
            self.assertEqual(temp, expected, "feature type (Vectors.sparse) or values (vector length or values) are incorrect")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass

    def test_aggregate_one_event_diagnostic(self):
        try:
            diags = self.spark.sparkContext.parallelize([Diagnostic("patient1", "code1", date.today())])
            actual = constructDiagnosticFeatureTuple(diags).collect()
            expected = [(('patient1', 'code1'), 1.0)]
            self.assertEqual(actual, expected, "Diagnostic: test aggregate one event failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass

    def test_aggregate_two_different_events_diagnostic(self):
        try:

            diags = self.spark.sparkContext.parallelize(
                [Diagnostic("patient1", "code1", date.today()),
                Diagnostic("patient1", "code2", date.today())])
            actual = constructDiagnosticFeatureTuple(diags).collect()
            expected = [(('patient1', 'code1'), 1.0), (('patient1', 'code2'), 1.0)]
            self.assertEqual(sorted(actual), sorted(expected), "Diagnostic: test aggregate two different events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass

    def test_aggregate_two_same_events_diagnostic(self):
        try:
            diags = self.spark.sparkContext.parallelize([
                Diagnostic("patient1", "code1", date.today()),
                Diagnostic("patient1", "code1", date.today())
            ])
            actual = constructDiagnosticFeatureTuple(diags).collect()
            expected = [(('patient1', 'code1'), 2.0)]
            self.assertEqual(actual, expected, "Diagnostic: test aggregate two same events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_three_events_with_duplication_diagnostic(self):
        try:
            diags = self.spark.sparkContext.parallelize([
                Diagnostic("patient1", "code1", date.today()),
                Diagnostic("patient1", "code1", date.today()),
                Diagnostic("patient1", "code2", date.today())
            ])
            actual = constructDiagnosticFeatureTuple(diags).collect()
            expected = [(('patient1', 'code1'), 2.0),
                        (('patient1', 'code2'), 1.0)]
            self.assertEqual(sorted(actual), sorted(expected), "Diagnostic: test aggregate three events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_filter_diagnostic(self):
        try:
            diags = self.spark.sparkContext.parallelize([
                Diagnostic("patient1", "code1", date.today()),
                Diagnostic("patient1", "code1", date.today()),
                Diagnostic("patient1", "code2", date.today())
            ])
            # Test filtering for code2
            actual_code2 = constructDiagnosticFeatureTuple(diags, {"code2"}).collect()
            expected_code2 = [(('patient1', 'code2'), 1.0)]
            self.assertEqual(actual_code2, expected_code2, "Diagnostic: test filter events for code2 failed")

            # Test filtering for code1
            actual_code1 = constructDiagnosticFeatureTuple(diags, {"code1"}).collect()
            expected_code1 = [(('patient1', 'code1'), 2.0)]
            self.assertEqual(actual_code1, expected_code1, "Diagnostic: test filter events for code1 failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_one_event_medication(self):
        try:
            meds = self.spark.sparkContext.parallelize([
                Medication("patient1", date.today(), "code1")
            ])
            actual = constructMedicationFeatureTuple(meds).collect()
            expected = [(('patient1', 'code1'), 1.0)]
            self.assertEqual(actual, expected, "Medication: test aggregate one event failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass

    def test_aggregate_two_different_events_medication(self):
        try:
            meds = self.spark.sparkContext.parallelize([
                Medication("patient1", date.today(), "code1"),
                Medication("patient1", date.today(), "code2")
            ])
            actual = constructMedicationFeatureTuple(meds).collect()
            expected = [(('patient1', 'code1'), 1.0),
                        (('patient1', 'code2'), 1.0)]
            self.assertEqual(sorted(actual), sorted(expected), "Medication: test aggregate two different events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_two_same_events_medication(self):
        try:
            meds = self.spark.sparkContext.parallelize([
                Medication("patient1", date.today(), "code1"),
                Medication("patient1", date.today(), "code1")
            ])
            actual = constructMedicationFeatureTuple(meds).collect()
            expected = [(('patient1', 'code1'), 2.0)]
            self.assertEqual(actual, expected, "Medication: test aggregate two same events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_three_events_with_duplication_medication(self):
        try:
            meds = self.spark.sparkContext.parallelize([
                Medication("patient1", date.today(), "code1"),
                Medication("patient1", date.today(), "code1"),
                Medication("patient1", date.today(), "code2")
            ])
            actual = constructMedicationFeatureTuple(meds).collect()
            expected = [(('patient1', 'code1'), 2.0),
                        (('patient1', 'code2'), 1.0)]
            self.assertEqual(sorted(actual), sorted(expected), "Medication: test aggregate three events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_filter_medication(self):
        try:
            meds = self.spark.sparkContext.parallelize([
                Medication("patient1", date.today(), "code1"),
                Medication("patient1", date.today(), "code1"),
                Medication("patient1", date.today(), "code2")
            ])
            # Test filtering for code2
            actual_code2 = constructMedicationFeatureTuple(meds, {"code2"}).collect()
            expected_code2 = [(('patient1', 'code2'), 1.0)]
            self.assertEqual(actual_code2, expected_code2, "Medication: test filter events for code2 failed")

            # Test filtering for code1
            actual_code1 = constructMedicationFeatureTuple(meds, {"code1"}).collect()
            expected_code1 = [(('patient1', 'code1'), 2.0)]
            self.assertEqual(actual_code1, expected_code1, "Medication: test filter events for code1 failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_one_event_lab(self):
        try:
            labs = self.spark.sparkContext.parallelize([
                LabResult("patient1", date.today(), "code1", 42.0)
            ])
            actual = constructLabFeatureTuple(labs).collect()
            expected = [(('patient1', 'code1'), 42.0)]
            self.assertEqual(actual, expected, "Labs: test aggregate one event failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_two_different_events_lab(self):
        try:
            labs = self.spark.sparkContext.parallelize([
                LabResult("patient1", date.today(), "code1", 42.0),
                LabResult("patient1", date.today(), "code2", 24.0)
            ])
            actual = constructLabFeatureTuple(labs).collect()
            expected = [(('patient1', 'code1'), 42.0),
                        (('patient1', 'code2'), 24.0)]
            self.assertEqual(sorted(actual), sorted(expected), "Labs: test aggregate two different events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_two_same_events_lab(self):
        try:
            labs = self.spark.sparkContext.parallelize([
                LabResult("patient1", date.today(), "code1", 42.0),
                LabResult("patient1", date.today(), "code1", 24.0)
            ])
            actual = constructLabFeatureTuple(labs).collect()
            expected = [(('patient1', 'code1'), 66.0 / 2)]  # Average value
            self.assertEqual(actual, expected, "Labs: test aggregate two same events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_aggregate_three_events_with_duplication_lab(self):
        try:
            labs = self.spark.sparkContext.parallelize([
                LabResult("patient1", date.today(), "code1", 42.0),
                LabResult("patient1", date.today(), "code1", 24.0),
                LabResult("patient1", date.today(), "code2", 7475.0)
            ])
            actual = constructLabFeatureTuple(labs).collect()
            expected = [(('patient1', 'code1'), 66.0 / 2),  # Average value
                        (('patient1', 'code2'), 7475.0)]
            self.assertEqual(sorted(actual), sorted(expected), "Labs: test aggregate three events failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
    def test_filter_lab(self):
        try:
            labs = self.spark.sparkContext.parallelize([
                LabResult("patient1", date.today(), "code1", 42.0),
                LabResult("patient1", date.today(), "code1", 24.0),
                LabResult("patient1", date.today(), "code2", 7475.0)
            ])
            # Test filtering for code2
            actual_code2 = constructLabFeatureTuple(labs, {"code2"}).collect()
            expected_code2 = [(('patient1', 'code2'), 7475.0)]
            self.assertEqual(actual_code2, expected_code2, "Labs: test filter events for code2 failed")

            # Test filtering for code1
            actual_code1 = constructLabFeatureTuple(labs, {"code1"}).collect()
            expected_code1 = [(('patient1', 'code1'), 66.0 / 2)]  # Average value
            self.assertEqual(actual_code1, expected_code1, "Labs: test filter events for code1 failed")
            FeatureConstructionTest.score += 1
        except AssertionError:
                    pass
# Add the remaining tests from your original script following the pattern of the above methods.

if __name__ == '__main__':

    # Create a test suite
    suite = unittest.TestLoader().loadTestsFromTestCase(FeatureConstructionTest)

    # Run the test suite
    unittest.TextTestRunner().run(suite)
    df_score.loc[df_score['Test'] == 'FeatureConstructionTest', 'Score'] = FeatureConstructionTest.score



# Evaluation Mertic [8 points]

In [ ]:
def getPurity(cluster_assignment_and_label):
    """
    TODO: base on the function to take the purity value
    param: cluster_assignment_and_label: rDD in the tuple format ((assigned_cluster_id, class), number)
    return: purity: the calculate purity value
    hints:
            Given input RDD with tuples of assigned cluster id by clustering,
            and corresponding real class. Calculate the getPurity of clustering.
            Purity is defined as
                        \fract{1}{N}\sum_K max_j |w_k \cap c_j|
            where N is the number of samples, K is the number of clusters, and j
            is the index of the class. w_k denotes the set of samples in the k-th cluster,
            and c_j denotes the set of samples of class j.
    """
    purity = None
    # start your code at here

    return purity

In [ ]:
import unittest
from pyspark.sql import SparkSession
from pyspark import SparkContext

class TestGetPurity(unittest.TestCase):
    score = 0

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder.appName("TestMetrics").getOrCreate()
        cls.spark.sparkContext.setLogLevel("ERROR")

        cls.test_input1 = cls.spark.sparkContext.parallelize([
            ((1, 1), 0),
            ((1, 2), 1),
            ((1, 3), 5),
            ((2, 1), 1),
            ((2, 2), 4),
            ((2, 3), 1),
            ((3, 1), 3),
            ((3, 2), 0),
            ((3, 3), 2)])

        cls.test_input2 = cls.spark.sparkContext.parallelize([
            ((1, 1), 0),
            ((1, 2), 53),
            ((1, 3), 10),
            ((2, 1), 0),
            ((2, 2), 1),
            ((2, 3), 60),
            ((3, 1), 0),
            ((3, 2), 16),
            ((3, 3), 0)])

    def test_get_purity_1(self):
        try:
            right_answer1 = (5 + 4 + 3) / 17.0
            student_purity1 = getPurity(self.test_input1)
            self.assertAlmostEqual(right_answer1, student_purity1)
            TestGetPurity.score += 4
        except AssertionError:
            pass

    def test_get_purity_2(self):
        try:
            right_answer2 = (53 + 60 + 16) / 140.0
            student_purity2 = getPurity(self.test_input2)
            self.assertAlmostEqual(right_answer2, student_purity2)
            TestGetPurity.score += 4
        except AssertionError:
            pass

    @classmethod
    def tearDownClass(cls):
        if TestGetPurity.score == 8:  # All tests passed
            print(f"All tests passed! Total Score: {TestGetPurity.score}/8")
        else:
            print(f"Total Score: {TestGetPurity.score}/8")

# Create a test suite
suite = unittest.TestLoader().loadTestsFromTestCase(TestGetPurity)

# Run the test suite
unittest.TextTestRunner().run(suite)
df_score.loc[df_score['Test'] == 'TestGetPurity', 'Score'] = TestGetPurity.score


# Clustering [25 points]


## Clustering coding part [15 points]

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.ml.feature import StandardScaler, PCA
from pyspark.ml.linalg import Vectors, DenseMatrix
from pyspark.ml.clustering import KMeans, GaussianMixture
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType


def clustering(phenotypeLabel, rawFeatures, k=3):

    print('phenotypeLabel: ', phenotypeLabel.count())
    standardizer = StandardScaler(withMean=True, withStd=True)
    df_features = rawFeatures.toDF(["id", "features"])
    scaler_model = standardizer.setInputCol("features").setOutputCol("scaled_features").fit(df_features)
    df_features = scaler_model.transform(df_features)

    # Using DataFrame operations to extract raw feature vectors
    raw_feature_vectors = df_features.select('features').rdd.map(lambda x: x[0]).cache()
    print('raw_feature_vectors: ', raw_feature_vectors)

    # Reduce dimension
    pca = PCA(k=10, inputCol="scaled_features", outputCol="pca_features")
    pca_model = pca.fit(df_features)
    df_features = pca_model.transform(df_features)

    # Ensure phenotypeLabel is a DataFrame
    if not isinstance(phenotypeLabel, DataFrame):
        phenotypeLabel = phenotypeLabel.toDF(["id", "label"])

    kmeans_purity = gmm_purity = None
    '''
    TODO: 1. K Means Clustering using pyspark ml
    Train a k means model using the variabe featureVectors as input and set seed as 6250

    TODO: 2. GMM Clustering using spark ml
    Train a Gaussian Mixture model using the variabe featureVectors as input and set seed as 6250

    TODO: 3. Compare clustering for the k = 3 case with the ground truth phenotypes that you computed for the rule-based PheKB algorithms.
    You can edit all the code as you wish and to generate all the table 1 and table 2's data.

    parameters:
                phenotypeLabel: an RDD or DataFrame containing phenotype labels. Each entry has a unique ID and an associated label.
                rawFeatures: an RDD containing raw feature vectors. Each entry has a unique ID and a corresponding feature vector.
    returns:
                kmeans_purity: purity values corresponding to the clustering results of KMean
                gmm_purity: purity values corresponding to the clustering results of Gaussian Mixture Model (GMM)
    hints:
                implement clustering using KMeans and compute the purity of the resulting clusters.
                implement clustering using Gaussian Mixture Model (GMM) and compute the purity of the resulting clusters.
                return the purity values for KMeans and GMM.
    Tips:       Pay attention to the variable features you need to calculate purity. 
                Consider which one you should use: features, scaled_features, or pca_features here. And Why?
    '''
    # start your code at here



    return kmeans_purity, gmm_purity

In [ ]:
import unittest
from pyspark.sql import SparkSession

class ClusteringTest(unittest.TestCase):
    score = 0

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder.appName("PhenotypingTest").getOrCreate()
        sc = cls.spark.sparkContext

        # Load the data output from the solution code
        phenotypeLabel = sc.textFile("data/phenotypeLabel.txt").map(lambda x: (x.split("\t")[0], int(x.split("\t")[1])))
        featureTuples = sc.textFile("data/featureTuples.txt").filter(lambda x: int(x[:9]) % 17 == 0).map(lambda x: ((x.split("\t")[0], x.split("\t")[1]), float(x.split("\t")[2])))
        # Convert tuples to vector using FeatureConstruction.construct solution
        rawFeatures = construct(featureTuples)

        # Run student solution to check KMeans, GaussianMixture if they can run properly
        cls.kMeansPurity, cls.gaussianMixturePurity = clustering(phenotypeLabel, rawFeatures)

    def test_kmeans_clustering(self):
        try:
            self.assertIsNotNone(self.kMeansPurity, "kMeansPurity should not be None")
            ClusteringTest.score += 7
        except AssertionError:
            pass

    def test_gmm_clustering(self):
        try:
            self.assertIsNotNone(self.gaussianMixturePurity, "gaussianMixturePurity should not be None")
            ClusteringTest.score += 7
        except AssertionError:
            pass

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()
        print(f"Purity of kMeans is: {ClusteringTest.kMeansPurity:.5f}")
        print(f"Purity of GMM is: {ClusteringTest.gaussianMixturePurity:.5f}")
        if ClusteringTest.score == 14:  # All tests passed
            ClusteringTest.score = ClusteringTest.score+1
            print(f"All tests passed! Total Score: {ClusteringTest.score}/15")
        else:
            print(f"Total Score: {ClusteringTest.score}/15")

# Create a test suite
suite = unittest.TestLoader().loadTestsFromTestCase(ClusteringTest)

# Run the test suite
unittest.TextTestRunner().run(suite)

df_score.loc[df_score['Test'] == 'ClusteringTest', 'Score'] = ClusteringTest.score



# The score summary of the coding part

In [ ]:
print(df_score)
print(f"The coding part score is: {df_score['Score'].sum()}")

## Clustering Writting part [10 points]

### KNN writting part [5 points]
| Percentage Cluster | Case | Control | Unknown |
|--------------------|------|---------|---------|
| Cluster 1          | x%   | y%      | z%      |
| Cluster 2          | xx%  | yy%     | zz%     |
| Cluster 3          | xxx% | yyy%    | zzz%    |
| **Total**          | **100%** | **100%** | **100%** |

Add your discussion:

### GMM writting part [5 points]
| Percentage Cluster | Case | Control | Unknown |
|--------------------|------|---------|---------|
| Cluster 1          | x%   | y%      | z%      |
| Cluster 2          | xx%  | yy%     | zz%     |
| Cluster 3          | xxx% | yyy%    | zzz%    |
| **Total**          | **100%** | **100%** | **100%** |

Add your discussion here:

# Discussion on K-means and GMM [10 points]
Base on the main function output the data you needed, you can modify the main function.


| k  | K-Means (All features) | K-Means (Filtered features) | GMM (All Features) | GMM (Filtered features) |
|----|------------------------|-----------------------------|--------------------|-------------------------|
| 2  |                        |                             |                    |                         |
| 5  |                        |                             |                    |                         |
| 10 |                        |                             |                    |                         |
| 15 |                        |                             |                    |
                         |


Add your discussion here:

## Code for discussion on K-means and GMM

In [ ]:
''''
This is for you to run and get the outcome for the Discussion on K-means and GMM part,
you can modify this code as you want get your results for the discussion,
fill the result into the above's figure

Tips: 
    If you have some doubts about the results of the discussion section or your results are not very good, 
    you can analyze good results during the discussion, which can still get good scores.
'''
def loadLocalRawData() -> Tuple[Set[str], Set[str], Set[str]]:

    with open("data/med_filter.txt") as f:
        candidateMedication = set(map(str.lower, f.read().splitlines()))

    with open("data/lab_filter.txt") as f:
        candidateLab = set(map(str.lower, f.read().splitlines()))

    with open("data/icd9_filter.txt") as f:
        candidateDiagnostic = set(map(str.lower, f.read().splitlines()))

    return (candidateMedication, candidateLab, candidateDiagnostic)

def main():
    spark = SparkSession.builder.getOrCreate()
    sc = spark.sparkContext

    # Set log levels
    logger = spark._jvm.org.apache.log4j
    logger.Level.WARN

    # Initialize loading of data
    medication, lab_result, diagnostic = load_rdd_raw_data(spark)
    candidate_medication, candidate_lab, candidate_diagnostic = loadLocalRawData()

    # Conduct phenotyping
    phenotype_label = transform(medication, lab_result, diagnostic)


    # Feature construction with all features
    feature_tuples = constructDiagnosticFeatureTuple(diagnostic).union(
        constructLabFeatureTuple(lab_result)
    ).union(
        constructMedicationFeatureTuple(medication)
    )



    rawFeatures = construct(feature_tuples)

    kMeansPurity, gaussianMixturePurity = clustering(phenotype_label, rawFeatures)

    print(f"[All feature] purity of kMeans is: {kMeansPurity:.5f}")
    print(f"[All feature] purity of GMM is: {gaussianMixturePurity:.5f}")


    filteredFeatureTuples = constructDiagnosticFeatureTuple(diagnostic, candidate_diagnostic).union(
        constructLabFeatureTuple(lab_result, candidate_lab)
    ).union(
        constructMedicationFeatureTuple(medication, candidate_medication)
    )

    filteredRawFeatures = construct(filteredFeatureTuples)


    kMeansPurity2, gaussianMixturePurity2 = clustering(phenotype_label, filteredRawFeatures)

    print(f"[Filtered feature] purity of kMeans is: {kMeansPurity2:.5f}")
    print(f"[Filtered feature] purity of GMM is: {gaussianMixturePurity2:.5f}")
main()

# Submmissions [5 points]

When submitting your work, ensure that you upload only the .ipynb file. It's crucial to execute all the cells in the notebook and display their results within the file. If the code outputs are not visible in the submitted .ipynb file, you may receive a score of 0/5 for that section. Additionally, for any written portions of the assignment, make sure to include them directly in the .ipynb file as well. This can be done by adding text cells where you can type your written responses or explanations.
Do not Change any of the test code!
